# 11 — Development arbitration and one-shot confirmation

Choose between the notebook-09 baseline and notebook-10 hybrid using stored
development results only. If neither clears its causal controls, stop without
opening confirmatory outcomes. Otherwise evaluate exactly one winner.

## 1. Setup

In [ ]:
EXTRAS = 'sim,analysis'
SETUP_ENV = False
import urllib.request
exec(urllib.request.urlopen('https://raw.githubusercontent.com/ArjunS07/cs159-sp26/main/pnp-vla/scripts/colab_bootstrap.py').read().decode())

## 2. Development-only arbitration

In [ ]:
import json, torch
from pnp import notebook as nb
from pnp.verifier import *

ctx=nb.setup("verifier_confirmation"); store,DEVICE,OUTPUT=ctx.store,ctx.device,ctx.output

def latest(experiment):
    rows=(store.client.table("verifier_models").select("verifier_id,created_at,metrics_json")
          .eq("experiment",experiment).order("created_at",desc=True).limit(1).execute().data or [])
    return rows[0] if rows else None

candidates=[]
for experiment in ("state-conditioned-verifier-v2","hybrid-chunk-critic-v1"):
    row=latest(experiment)
    if not row: continue
    checkpoint,_=store.load_verifier(row["verifier_id"])
    metadata=checkpoint.get("metadata",{})
    report=(metadata.get("development_report") or
            (row.get("metrics_json") or {}).get("development_report") or
            row.get("metrics_json") or {})
    metrics=report.get("metrics") or report.get("development_metrics") or {}
    gate=report.get("gate") or report.get("registration_gate") or {}
    candidates.append({"experiment":experiment,"row":row,"checkpoint":checkpoint,
                       "metadata":metadata,"metrics":metrics,
                       "eligible":bool(gate.get("eligible",False))})

eligible=[c for c in candidates if c["eligible"]]
assert eligible, "No development model cleared its causal-control gate; confirmation stays sealed."
best_ranking=max(c["metrics"].get("group_macro_ranking_accuracy",-1) for c in eligible)
statistically_tied=[c for c in eligible
                    if best_ranking-c["metrics"].get("group_macro_ranking_accuracy",-1)<=.005]
winner=min(statistically_tied,key=lambda c:c["metadata"].get("parameter_count",10**20))
print({"winner":winner["experiment"],"verifier_id":winner["row"]["verifier_id"],
       "development_ranking":winner["metrics"].get("group_macro_ranking_accuracy")})

## 3. Open the sealed outcomes once and evaluate the winner

In [ ]:
examples=load_candidate_examples(
    store,"verifier-v2-pro-confirmatory",cache_dir=OUTPUT/"confirmatory_cache")
audit=validate_candidate_groups(examples,expected_candidates=12)
assert audit["groups"]>=150,audit
checkpoint=winner["checkpoint"]
if winner["experiment"]=="hybrid-chunk-critic-v1":
    model=HybridChunkCritic(**checkpoint["architecture"])
    model.load_state_dict(checkpoint["state_dict"])
else:
    spec=checkpoint["metadata"]["selected_spec"]
    model=CompactAdvantageVerifier(action_width=64,dropout=spec["dropout"],
                                   conditioning=spec["architecture"])
    model.load_state_dict(checkpoint["model"])
model=model.to(DEVICE).eval()
metrics=evaluate_candidate_ranker(
    model,examples,DEVICE,config=AdvantageTrainConfig(seed=20260729,prefix_length=10))
report={"winner":winner["experiment"],"verifier_id":winner["row"]["verifier_id"],
        "integrity":audit,"confirmatory_metrics":metrics}
(OUTPUT/"confirmatory_report.json").write_text(json.dumps(report,indent=2,sort_keys=True))
(store.client.table("verifier_models").update({"metrics_json":report})
 .eq("verifier_id",winner["row"]["verifier_id"]).execute())
print(json.dumps(report,indent=2))